# Generator lab

One place to drive, compare and listen to every drone-noise generator in the
project. Replaces `drone_embedding_explorer`, `noise_gen_real_vs_generated`,
`noise_four_way_comparison` and `jasa_gp_interactive` — several of which no
longer import, because they still reach for `data_processing.dregon` /
`.michaels`, which the data-layer refactor moved into `data_processing.sources`.

What you can do here:

1. **Pick one or more generators** — learned (`deep/*`), the JASA GP, the CONA
   constant-RPS auralization, or the real recording as reference.
2. **Pick an excitation** — a window of any real recording (audio comes with it,
   so you can A/B), or a synthetic RPS trajectory when you want a condition the
   data does not contain.
3. **Move the model's parameters live** — the per-drone embedding (including
   *between* and *beyond* the two learned codes), the RPS-jitter linewidth, and
   the wind channel on/off — and watch spectrograms, a spectrum slice under a
   time slider, and audio update together.

All logic lives in `notebooks/generator_lab.py`; this notebook is only the
controls, so the same functions are usable from a script or a report.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in (ROOT, ROOT / "src", ROOT / "notebooks"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import warnings

import ipywidgets as W
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import generator_lab as lab

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})

print(f"{len(lab.VARIANTS)} variants available:")
for name, spec in lab.VARIANTS.items():
    print(f"  [{spec.family:5s}] {name}")

12 variants available:
  [real ] real
  [deep ] deep/e6-perdrone (old geometry)
  [deep ] deep/v1 corrected-geometry
  [deep ] deep/v1 recalibrated labels
  [deep ] deep/v1 recal, 8 mics
  [deep ] deep/likelihood, 8 mics
  [deep ] deep/likelihood + wind
  [deep ] deep/spatial + wind
  [deep ] deep/spatial + wind, uniform
  [gp   ] gp/JASA rotor field
  [cona ] cona/constant-RPS auralization
  [fwh  ] fwh/physics simulator


## 1 · Choose an excitation

Either a slice of a real recording, or a synthetic RPS trajectory.

A real slice carries its own audio and geometry, so `real` becomes available as
a comparison row. A synthetic one does not — it exists to probe conditions the
recordings do not cover (a full takeoff-to-landing envelope, an aggressive
manoeuvre, a rotor speed nothing was recorded at).

In [5]:
src_kind = W.ToggleButtons(options=["real recording", "synthetic RPS"], value="real recording",
                           description="source:")
dataset  = W.Dropdown(options=list(lab.DATASETS), description="dataset:")
rec      = W.Dropdown(options=lab.recordings(dataset.value), description="recording:")
start_s  = W.FloatSlider(value=30.0, min=0.0, max=300.0, step=0.5, description="start (s):",
                         continuous_update=False, readout_format=".1f")
dur_s    = W.FloatSlider(value=4.0, min=1.0, max=10.0, step=0.5, description="length (s):",
                         continuous_update=False, readout_format=".1f")
syn_kind = W.Dropdown(options=["intermittent", "full_flight"], description="trajectory:")
syn_drone= W.Dropdown(options=["dregon", "michaels"], description="rig:")
syn_seed = W.IntSlider(value=0, min=0, max=99, description="seed:", continuous_update=False)

def _on_dataset(change):
    rec.options = lab.recordings(change["new"])
dataset.observe(_on_dataset, names="value")

real_box = W.VBox([dataset, rec, start_s, dur_s])
syn_box  = W.VBox([syn_drone, syn_kind, syn_seed, dur_s])
src_area = W.VBox([real_box])

def _on_kind(change):
    src_area.children = [real_box] if change["new"] == "real recording" else [syn_box]
src_kind.observe(_on_kind, names="value")

display(W.VBox([src_kind, src_area]))

In [11]:
def current_excitation() -> lab.Excitation:
    if src_kind.value == "real recording":
        return lab.real_slice(dataset.value, rec.value, start_s.value, dur_s.value)
    return lab.synth_slice(drone=syn_drone.value, kind=syn_kind.value,
                           dur_s=dur_s.value, seed=syn_seed.value)

exc = current_excitation()
print(exc.label)
print(f"  drone      {exc.drone}")
print(f"  mics       {exc.mic_pos.shape[0]}   rotors {exc.rotor_pos.shape[0]}")
print(f"  duration   {exc.duration_s:.2f}s   mean RPS {exc.mean_rps:.1f} rev/s")
print(f"  per-rotor  {np.round(exc.rps.mean(axis=-1), 1).tolist()} rev/s")
print(f"  reference audio: {'yes' if exc.audio is not None else 'no (synthetic)'}")

dict_keys(['audio', 'motors_command', 'imu_accel', 'imu_gyro', 'mic_pos', 'rotor_pos', 'meta', 'motors_command_raw', 'audio_timestamps'])
GridIndex(sr_num=44100, size=3599989, sr_den=1, t_start_ticks=1512727385205045504, dur_ticks=81632403628, phase=0.0)


AttributeError: 'GridIndex' object has no attribute 'step'

## 2 · Choose generators and move their parameters

`alpha` walks the per-drone embedding: **0 = DREGON, 1 = Michael's**. Values
outside that range extrapolate, and `offset` pushes off the axis entirely — both
land in regions no training data constrains, which is exactly what makes them
worth listening to.

`jitter` overrides the learned RPS-jitter linewidth (the mechanism that
broadens the harmonic comb; the E6 round found it is what real recordings need).
Leave it at `-1` to use whatever the model learned.

`wind` zeroes the wind channel on variants that have one — the cheapest way to
hear what it actually contributes. On the marginal-likelihood variant it should
be inaudible, because that channel trained to ~0.1% of predicted power.

In [ ]:
picker = W.SelectMultiple(options=list(lab.VARIANTS), value=("real", "deep/likelihood, 8 mics"),
                          rows=11, description="variants:", layout=W.Layout(width="640px"))
alpha  = W.FloatSlider(value=0.0, min=-1.0, max=2.0, step=0.05, description="alpha:",
                       continuous_update=False, readout_format=".2f")
use_alpha = W.Checkbox(value=False, description="override embedding")
offset = W.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description="offset:",
                       continuous_update=False, readout_format=".2f")
jitter = W.FloatSlider(value=-1.0, min=-1.0, max=2.0, step=0.05, description="jitter:",
                       continuous_update=False, readout_format=".2f")
wind   = W.Checkbox(value=True, description="wind channel on")
mic    = W.IntSlider(value=0, min=0, max=7, description="mic:", continuous_update=False)
norm   = W.Checkbox(value=True, description="match RMS to reference")

display(W.HBox([picker, W.VBox([use_alpha, alpha, offset, jitter, wind, mic, norm])]))

## 3 · Render, look, listen

Spectrograms for every selected variant, a time slider that pulls out a single
spectrum column as a line plot, and an audio player per row.

Level and spectral *shape* are separate claims, and these families do not agree
on absolute gain — CONA and the GP carry their own calibration. **match RMS**
makes the shape comparison honest; untick it when you care about level.

In [ ]:
_cache: dict = {}

def render_all():
    exc = current_excitation()
    a = alpha.value if use_alpha.value else None
    j = None if jitter.value < 0 else jitter.value
    rows = {}
    for name in picker.value:
        key = (name, exc.label, a, offset.value, j, wind.value)
        if key not in _cache:
            try:
                _cache[key] = lab.render(name, exc, alpha=a, offset=offset.value,
                                         jitter_sigma=j, wind=wind.value)
            except Exception as e:                      # a missing checkpoint or
                _cache[key] = e                          # an unsupported family
        rows[name] = _cache[key]
    return exc, rows

def show(t_frac=0.5):
    exc, rows = render_all()
    ok = {k: v for k, v in rows.items() if isinstance(v, np.ndarray)}
    bad = {k: v for k, v in rows.items() if not isinstance(v, np.ndarray)}
    for k, e in bad.items():
        print(f"[skipped] {k}: {type(e).__name__}: {e}")
    if not ok:
        return
    ref = exc.audio[mic.value] if exc.audio is not None else next(iter(ok.values()))[mic.value]

    n = len(ok)
    fig, axes = plt.subplots(n, 2, figsize=(13, 2.5 * n), squeeze=False,
                             gridspec_kw={"width_ratios": [2.2, 1]})
    for i, (name, audio) in enumerate(ok.items()):
        ch = audio[min(mic.value, audio.shape[0] - 1)]
        if norm.value:
            ch = lab.match_rms(ch, ref)
        db, freqs, times = lab.spectrogram(ch)
        t = float(t_frac) * times[-1]
        ax = axes[i][0]
        ax.imshow(db, origin="lower", aspect="auto", cmap="magma", vmin=-80, vmax=0,
                  extent=[times[0], times[-1], freqs[0], freqs[-1]])
        ax.axvline(t, color="cyan", lw=1.0)
        ax.set_ylim(0, 4000); ax.set_ylabel("Hz")
        ax.set_title(f"{name}   (RMS {20*np.log10(np.sqrt(np.mean(ch**2))+1e-12):+.1f} dB)",
                     fontsize=9, loc="left")
        if i < n - 1: ax.set_xticklabels([])
        else: ax.set_xlabel("time (s)")
        axl = axes[i][1]
        axl.plot(freqs, lab.spectrum_at(db, times, t), lw=0.9)
        axl.set_xlim(0, 4000); axl.set_ylim(-90, 5)
        axl.set_ylabel("dB"); axl.grid(alpha=0.25)
        if i < n - 1: axl.set_xticklabels([])
        else: axl.set_xlabel("Hz")
    fig.suptitle(f"{exc.label}  ·  mic {mic.value}  ·  t = {t:.2f}s", fontsize=10)
    fig.tight_layout()
    plt.show()

    for name, audio in ok.items():
        ch = audio[min(mic.value, audio.shape[0] - 1)]
        if norm.value:
            ch = lab.match_rms(ch, ref)
        print(name)
        display(lab.player(ch))

W.interact(show, t_frac=W.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01,
                                      description="spectrum @", continuous_update=False));

## 4 · Why does a magnitude-trained model score badly on likelihood?

The 8-microphone magnitude model has a much worse held-out NLL than the 1-mic
one (10.918 vs 2.899 on DREGON), which looks paradoxical. The decomposition
below is the answer, and it is worth running rather than taking on trust.

The likelihood has two terms — `log(sigma^2)`, paying for claimed power, and
`(r-a)^2/sigma^2`, paying for surprise. A model trained with a magnitude loss
**never optimised either**. Its predicted variance is whatever its broadband
branch happened to settle on, so its NLL measures how badly calibrated an
uncalibrated model happens to be. That is close to an accident, and it is *not*
evidence that eight microphones hurt.

The honest comparison between those two models is on the criterion they were
actually trained for, where the gap is far smaller (mrstft 4.734 vs 3.602).

In [ ]:
import torch

from losses.spectral_likelihood import rice_nll
from tasks.noise_generation import geometry_to_rel_pos


def nll_terms(name: str, exc: lab.Excitation, n_fft: int = 1024) -> dict:
    """Split the held-out NLL into its two terms for one variant.

    `log(sigma^2)` pays for claimed power; `(r-a)^2/sigma^2` pays for surprise.
    A magnitude-trained model never optimised either, so its total is a measure
    of accidental calibration rather than of fit.
    """
    model = lab.load_variant(name)
    rps = torch.from_numpy(exc.rps).unsqueeze(0)
    rel = geometry_to_rel_pos(
        torch.from_numpy(exc.mic_pos).unsqueeze(0),
        torch.from_numpy(exc.rotor_pos).unsqueeze(0),
    )
    with torch.no_grad():
        stats = model.spectral_stats(rps, rel, [exc.drone])

        win = torch.hann_window(n_fft)

        def mag(x):
            x = x.reshape(-1, x.shape[-1])
            return torch.stft(x, n_fft=n_fft, hop_length=n_fft // 4, window=win,
                              return_complex=True, center=True).abs()

        r = mag(torch.from_numpy(exc.audio))            # [M, F, N]
        a = mag(stats["coherent"][0])
        psd = stats["noise_psd"][0]                      # [M, frames, freqs]
        psd = torch.nn.functional.interpolate(
            psd.unsqueeze(1), size=(r.shape[-1], r.shape[-2]),
            mode="bilinear", align_corners=True,
        ).squeeze(1).transpose(-1, -2)                   # -> [M, F, N]
        sigma2 = psd.clamp_min(0) * win.pow(2).sum() + 1e-4 * r.pow(2).mean()
        return {
            "log sigma^2": float(sigma2.log().mean()),
            "(r-a)^2/sigma^2": float(((r - a).pow(2) / sigma2).mean()),
            "total NLL": float(rice_nll(r, a, sigma2).mean()),
        }


exc = current_excitation()
if exc.audio is None:
    print("pick a REAL recording above — this needs reference audio")
else:
    for name in ("deep/v1 recalibrated labels", "deep/v1 recal, 8 mics",
                 "deep/likelihood, 8 mics"):
        try:
            t = nll_terms(name, exc)
            print(f"{name:34s} " + "  ".join(f"{k} {v:9.3f}" for k, v in t.items()))
        except Exception as e:
            print(f"{name:34s} skipped: {type(e).__name__}: {e}")